# E05：文本文件读取、清洗与结构化

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 with open 和 readlines 读取课程提供的 UTF-8 文本文件。
2. 使用 strip、split 和 replace 处理受控的 ASCII 文本问题。
3. 复用函数、循环、判断和异常处理，把文本行转换为结构化记录。
4. 检查记录数、字段、重复值和缺失值，并把结果保存为 JSON。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不填写真实调查信息，不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查结果是否与任务要求一致。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的数据设置单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- 如果变量状态混乱，重启内核并从上到下重新运行。
- E02–E04 的同源样本由课程准备代码载入；直接运行即可，不要求解释其中的文件读取代码。

## 建议节奏

- 路径确认、读取和观察：10 分钟
- 单行拆分和字段函数：15 分钟
- 记录解析和批量去重：20 分钟
- 类别、缺失值和记录数核验：15 分钟
- JSON 保存和官方检查点比较：15 分钟
- 调试、实验小结与提交：15 分钟

## 本章第一次读取外部文件

课程数据统一使用 UTF-8；代码中固定写 `encoding="utf-8"`，本实验不展开编码转换。无论从 Notebook 目录还是仓库根目录启动，输出都固定写入 `lab/applied-stat-64/chapter2/e05_output`，不得覆盖 `lab/shared`。

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

working_dir = Path.cwd()
course_chapter2 = working_dir / "lab" / "applied-stat-64" / "chapter2"
chapter2_dir = course_chapter2 if course_chapter2.is_dir() else working_dir

# 路径拼接在盘符根目录也安全，不按下标访问固定层数的父目录。
shared_dirs = [chapter2_dir / "../../shared", working_dir / "lab/shared"]

raw_candidates = [directory / "raw/student_survey_raw.txt" for directory in shared_dirs]
official_candidates = [directory / "checkpoints/e05_cleaned_records.json" for directory in shared_dirs]
raw_candidates.extend([chapter2_dir / "student_survey_raw.txt", working_dir / "student_survey_raw.txt"])
official_candidates.extend([chapter2_dir / "e05_cleaned_records.json", working_dir / "e05_cleaned_records.json"])
raw_path = next((path for path in raw_candidates if path.exists()), chapter2_dir / "student_survey_raw.txt")
official_path = next((path for path in official_candidates if path.exists()), chapter2_dir / "e05_cleaned_records.json")
output_path = chapter2_dir / "e05_output" / "e05_cleaned_records.json"

release_url = "https://rwang0040.github.io/python-for-statistics-public/lab/shared"

# 只下载本地缺少的文件，不覆盖已有输入；下载与保存失败分别提示。
for data_path, relative_url in [
    (raw_path, "/raw/student_survey_raw.txt"),
    (official_path, "/checkpoints/e05_cleaned_records.json"),
]:
    if not data_path.exists():
        try:
            with urlopen(release_url + relative_url, timeout=10) as response:
                data_bytes = response.read()
        except OSError as error:
            raise RuntimeError(
                f"无法联网下载 {data_path.name}。请检查网络，或手动下载到 {data_path.parent.resolve()}。"
            ) from error
        try:
            data_path.write_bytes(data_bytes)
        except OSError as error:
            raise RuntimeError(
                f"已下载 {data_path.name}，但无法保存到 {data_path.resolve()}。请检查写入权限或磁盘空间。"
            ) from error

print("原始文件：", raw_path.resolve())
print("官方检查点：", official_path.resolve())
print("当前工作目录：", working_dir)
print("文件是否存在：", raw_path.exists())
print("固定输出位置：", output_path.resolve())

## 任务 1：读取并观察原始文本

使用准备单元给出的 `raw_path`，以 `encoding="utf-8"` 打开文件，用 `readlines()` 保存为 `lines`。输出包含表头的总行数、表头 `lines[0]` 和前两条数据行 `lines[1]`、`lines[2]`。

In [ ]:
lines = []

# TODO：使用 with open 和 readlines 替换上面的空列表
# with open(...):
#     lines = ...

# TODO：输出总行数、表头和前两条数据行

## 任务 2：手工拆分一条记录

对第一条数据行先 `strip()`，再按逗号 `split()`，检查字段数量及各字段内容。

In [ ]:
# TODO：清理并拆分第一条数据行
# first_line =
# first_parts =
# print(...)

## 任务 3：完成字段清洗函数

先阅读课程提供的 `clean_text()`、`clean_category()` 和四个类别映射字典。它们负责去除首尾空白、识别缺失标记，以及按小写键映射规范类别。

在 `parse_number(value, percent=False)` 中补全浮点数转换与返回步骤。该函数先清理文本；`percent=True` 时还会移除百分号；缺失标记或转换失败时返回 `None`。

分别调用函数，核对以下结果：`" F "` 映射为 `"female"`，`"applied statistics"` 映射为 `"applied_statistics"`，`"71.7%"` 在百分数模式下得到 `71.7`，`"unknown"` 得到 `None`。

In [ ]:
missing_markers = ["", "na", "n/a", "none", "unknown", "-"]


def clean_text(value):
    text = value.strip()
    if text.lower() in missing_markers:
        return None
    return text


def clean_category(value, mapping):
    text = clean_text(value)
    if text is None:
        return None
    key = text.lower()
    if key in mapping:
        return mapping[key]
    return None


def parse_number(value, percent=False):
    text = clean_text(value)
    if text is None:
        return None
    if percent:
        text = text.replace("%", "").strip()
    try:
        # TODO：把清理后的文本转换为浮点数并返回
        return None
    except ValueError:
        return None


gender_mapping = {
    "female": "female",
    "f": "female",
    "male": "male",
    "m": "male",
}
major_mapping = {
    "statistical_computing": "statistical_computing",
    "applied_statistics": "applied_statistics",
    "applied statistics": "applied_statistics",
    "data_science": "data_science",
    "data science": "data_science",
}
prior_mapping = {"yes": "yes", "y": "yes", "no": "no", "n": "no"}
plan_mapping = {
    "regular": "regular",
    "standard": "regular",
    "feedback": "feedback",
    "instant_feedback": "feedback",
}

# TODO：用正常值、类别别名、百分数和缺失标记检查函数

## 任务 4：把一行文本解析成一条记录

补全 `parse_record(line)` 的记录字典。课程已提供前两个字段，按下面顺序补全其余六个字段的清洗调用：

| 位置 | 字段 | 清洗函数 |
| --- | --- | --- |
| 0 | `student_id` | `clean_text()`，课程已提供 |
| 1 | `alias` | `clean_text()`，课程已提供 |
| 2 | `gender` | `clean_category()`，使用 `gender_mapping` |
| 3 | `major` | `clean_category()`，使用 `major_mapping` |
| 4 | `prior_programming` | `clean_category()`，使用 `prior_mapping` |
| 5 | `support_plan` | `clean_category()`，使用 `plan_mapping` |
| 6 | `reported_study_hours` | `parse_number()` |
| 7 | `reported_attendance_rate` | `parse_number()`，启用百分数模式 |

字段数不是 8 时，按课程提供的检查返回 `None`。用第一条数据行 `lines[1]` 调用函数并输出结果。

In [ ]:
def parse_record(line):
    parts = line.strip().split(",")
    if len(parts) != 8:
        return None

    return {
        "student_id": clean_text(parts[0]),
        "alias": clean_text(parts[1]),
        # TODO：参照字段顺序补全其余 6 个字段的清洗调用
    }


# TODO：调用 parse_record() 检查第一条数据行

## 任务 5：批量生成结构化记录并去重

遍历 `lines[1:]`，跳过表头，对每行调用 `parse_record()`，补全课程提供的分支：

1. 解析结果为 `None` 或清理后的 `student_id` 为 `None`，将 `invalid_line_count` 加 1。
2. 学号已在 `seen_ids` 中，将 `duplicate_count` 加 1，保留该学号第一次出现的有效记录。
3. 其他情况，将学号加入 `seen_ids`，完整记录加入 `clean_records`，保持原顺序。

数值字段缺失的记录仍可保留。输出清洗后记录数、重复记录数和无法解析的记录数。

In [ ]:
clean_records = []
seen_ids = []
invalid_line_count = 0
duplicate_count = 0

for line in lines[1:]:
    record = parse_record(line)
    if record is None or record["student_id"] is None:
        # TODO：更新无法解析的记录数
        pass
    elif record["student_id"] in seen_ids:
        # TODO：更新重复记录数
        pass
    else:
        # TODO：保存 student_id 和记录
        pass

# TODO：输出清洗后记录数、重复记录数和无法解析的记录数

## 任务 6：检查类别和缺失值

遍历 `clean_records`，分别统计 `reported_study_hours` 和 `reported_attendance_rate` 为 `None` 的数量，保存为 `missing_hours` 和 `missing_attendance`。

分别收集 `gender`、`major`、`support_plan` 的不重复取值，保存到课程提供的 `genders`、`majors`、`plans` 列表。输出两项缺失数及排序后的三个类别列表。

In [ ]:
missing_hours = 0
missing_attendance = 0
genders = []
majors = []
plans = []

for record in clean_records:
    # TODO：统计两个数值字段的缺失数量
    # TODO：分别收集三个类别字段的不重复规范值
    pass

# TODO：输出缺失数量和排序后的类别列表

## 任务 7：保存学生输出

运行课程提供的目录准备代码，把 `clean_records` 以 UTF-8 编码写为 JSON，路径使用准备单元给出的 `output_path`。再从同一路径读回，保存为 `saved_records`。

输出文件的实际路径和重新读取的记录数，供下一任务核验。

In [ ]:
# 路径准备代码由课程提供；本任务只练习写入和重新读取
output_path.parent.mkdir(exist_ok=True)

# TODO：把 clean_records 写为 JSON，再重新读取
# with open(...):
#     ...

# print(...)

## 任务 8：与官方检查点核验

以 UTF-8 编码只读打开准备单元给出的 `official_path`，将 JSON 载入 `official_records`。

输出官方记录数，并与任务 7 的 `saved_records` 比较：第一条记录的字段名称及顺序是否一致，全部记录列表是否相等。列表比较同时检查记录内容和顺序。官方文件保持只读。

In [ ]:
# TODO：读取 official_path 并比较记录数、字段和全部记录
# 注意：只读取，不覆盖官方检查点

## 实验小结

请用 Markdown 回答：

1. 文件读取成功为什么不等于数据已经可以分析？
2. E02–E04 中的哪些知识在本实验中被重新使用？
3. 为什么学生输出和官方检查点必须分开保存？

### 实验小结

- 